In [2]:

import os
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
load_dotenv()

llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    temperature=0,
    api_key=os.environ["XIAOMI_API_KEY"],
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
)

prompt = ChatPromptTemplate.from_template("讲一个关于{topic}的笑话，不要有任何解释")
chain = prompt | llm | StrOutputParser()

res = chain.invoke({"topic": "AI"})
print(res)


async for chunk in chain.astream({"topic": "parrot"}):
    print(chunk, end="|", flush=True)

一个AI走进酒吧，点了一杯啤酒。

酒保问："你要小杯还是大杯？"

AI说："是的。"
|||||||一个人|走进|宠物店，|看到一只|鹦鹉站在|两|根树枝|上，|脚上|还|挂着|标|价：|2|000元|。

他|好奇|地问老板|："这只|鹦鹉怎么|这么贵？|"

老板说|："它会说|两种语言。"

|他|看到|旁边另一|只鹦|鹉，|标价50|00元|，又|问："那|这只呢？"

|老板说："|它会说|五种语言。|"

这时|他注意到|角落里有|只又老|又丑|的鹦鹉，|毛都快掉|光了，标|价竟然|要|2|0000|元。

他吓|了一跳："这只|凭什么|这么|贵？！"

|老板压|低声音说："|说实话|，我也不知道它|会什么|，但另外|两只都|管它叫'|老板'。"||||

In [11]:
from langchain_core.runnables import RunnableParallel
from langchain_core.runnables import chain


joke_chain = ChatPromptTemplate.from_template("给我讲一个关于{topic}的笑话") | llm
poem_chain = ChatPromptTemplate.from_template("给我写一首关于{topic}的绝句") | llm

map_chain = RunnableParallel(joke=joke_chain, poem=poem_chain)
# map_chain.invoke({"topic": "程序员"})


prompt1 = ChatPromptTemplate.from_template("tell me a joke about {topic}")
prompt2 = ChatPromptTemplate.from_template("what is the subject of this joke :{joke}")

@chain
def custom_chain(text):
  prompt_vall = prompt1.invoke({"topic": text})
  output1 = llm.invoke(prompt_vall)

  parsed_output1 = StrOutputParser().invoke(output1)
  
  chain2 = prompt2 | llm | StrOutputParser()
  return chain2.invoke({"joke": parsed_output1})


custom_chain.invoke("bears")

'The subject of the joke is **bears** – specifically, their natural fur coats. The humor comes from imagining a bear trying to wear human clothing like a denim jacket. 🐻'

In [ ]:
from langchain_core.runnables import RunnablePassthrough


runnable = RunnableParallel(
  passed=RunnablePassthrough(),
  modified=lambda x: x["num"] + 1,
)

runnable.invoke({"num": 1})

{'passed': {'num': 1}, 'modified': 2}

In [ ]:
from langchain_core.messages import AIMessage, BaseMessage
from langchain_core.messages import BaseMessage
from langchain_core.chat_history import BaseChatMessageHistory
from pydantic import BaseModel, Field


class InMemoryHistory(BaseChatMessageHistory, BaseModel):
  """内存中实现聊天记录历史消息"""
  messages: list[BaseMessage] = Field(default_factory=list)

  def add_message(self, message: BaseMessage) -> None:
    """添加消息到历史记录"""
    self.messages.append(message)

  def clear(self) -> None:
    """清空历史记录"""
    self.messages = []

store={}

def get_by_session_id(session_id: str) -> BaseChatMessageHistory:
  """根据session_id获取聊天记录"""
  if session_id not in store:
    store[session_id] = InMemoryHistory()
  return store[session_id]

history = get_by_session_id("1")
history.add_ai_message(AIMessage(content="你好"))
print(store)



{'1': InMemoryHistory(messages=[AIMessage(content='你好', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])}
